In [1]:
%run Latex_macros.ipynb

<IPython.core.display.Latex object>

# Before you spend GPU: a pre-flight

## Where this session fits

> **The limitation we start from: a GRPO run costs hours of GPU, and a group whose samples all earn one reward teaches nothing.**

- **Words used here**
    - **group:** the $G$ samples drawn for one prompt; **$G$** is the group size
    - **checkpoint:** the saved model after SFT, where the GRPO run starts
    - **hit:** a sample that earns reward (C1)
- **The question:** how do we check cheaply, before the run, whether it can learn, and which $G$ to use?
- **Position:** a companion to Intro C1, C2 and Part 1 of the assignment
    - optional; not a numbered RLLM session
- what we did and why, not claimed to be the best recipe

**By the end you can**

1. say why a run can have no **signal** (a nonzero advantage on some completion of a group)
2. compute the chance that a group of $G$ samples contains a success and a failure
3. read a small table of group statistics
4. list two other ways to decide whether a run can start
5. write a decision record: decision, criterion, evidence, outcome
- next: the map of the session


## Map: the run and its pre-flight

> **The pre-flight is cheap sampling on the SFT checkpoint before the long run. We use it for three decisions.**

<img src="images/p1_map.png" width=70%>

- the assignment's three decisions, in its words
    1. Is the SFT checkpoint good enough to start GRPO? (**Exploration**)
    2. Which $G$? (**Group size**, measured in **Sweep**)
    3. Go or no-go for the full run? It combines 1 and 2, so it has no section of its own: see **Decision record**
- **Signal** is the background for 1 and 2; **Other ways** can serve all three


## Signal: no variance, no signal

**Why this section?** The pre-flight asks one thing: will the groups carry signal?

> **A group whose rewards are all equal gives every completion advantage 0. Only groups that differ teach anything.**

- **From C2:** if all four samples succeed, or all four fail, every advantage is $0$
    - the KL penalty only pulls toward the **reference model** (the frozen start)
- **signal:** a nonzero advantage on at least one completion of a group
- with a binary reward, a group carries signal exactly when it holds a hit and a miss
- we can sample the checkpoint before the run to see whether groups differ (our practice)


## Signal: the chance a group has both outcomes

> **With success probability $p$ and independent samples, a group of $G$ has both outcomes with probability $1 - p^G - (1-p)^G$.**

- all successes: $p^G$; all failures: $(1-p)^G$; subtract both from $1$
- hand check: $p = 0.05$, $G = 8$ gives $\approx 0.337$

| success probability p | G = 2 | G = 4 | G = 8 | G = 16 |
|:--|--:|--:|--:|--:|
| 0.05 | 0.095 | 0.185 | 0.337 | 0.560 |
| 0.1 | 0.180 | 0.344 | 0.570 | 0.815 |
| 0.3 | 0.420 | 0.752 | 0.942 | 0.997 |
| 0.5 | 0.500 | 0.875 | 0.992 | 1.000 |

- computed from the formula, not measured; in a real run $p$ varies by prompt
- with a **graded reward** ties are rarer, so we read the group's standard deviation instead
    - **our reading** (our judgment, untested): spread below the noise points nowhere


## Exploration: does the checkpoint ever succeed?

> **For each skill the reward needs, we measured how often the checkpoint succeeds. A rate near zero leaves RL nothing to amplify (C1).**

<img src="images/p1_explore.png" width=84%>

- a **skill** is one separate thing the reward needs: a format, a method, a final step
- our practice: sample at **temperature** 1 (no sharpening) from the real checkpoint
    - score each skill with its own check
- the assignment's exploration check (`base_rate.py`) looks at three skills: `sigma` and `rate` (together, extraction accuracy) and `chain` (chain-resolution rate)
    - the figure's four are illustrative
- invented: the last skill succeeds at 0.01, so a group of 8 has a hit with chance 0.08
    - our reading: we would return to SFT


## Group size: signal grows with G, and so does cost

> **Signal rises with $G$; cost rises in a straight line. Our practice: the smallest $G$ whose groups are mostly informative.**

<img src="images/p1_curve.png" width=68%>

- computed for three invented values of $p$ and an invented target of 0.8
    - the smallest $G$ is 5 for p = 0.3, 16 for p = 0.1, 32 for p = 0.05
- the curve approaches 1, so each extra sample buys less
- **our reading: an inherited $G$ is a risk**
    - it was tuned for another $p$, so we measured $G$ on our own checkpoint


## Sweep: three columns we used

> **The given `readiness_sweep` helper samples groups at several $G$ and returns three columns. We found them useful; others are possible.**

1. **parse_rate:** the fraction of samples the answer extractor parses (here, the chain-resolution rate)
    - an unparsable sample earns no reward
2. **within_group_std_mean:** each prompt's group reward std, averaged over prompts
    - divisor $G - 1$ as in C2: our choice
3. **frac_all_tied:** the fraction of groups whose rewards are all equal, so with no signal

**Ready** is our word for a checkpoint and $G$ judged able to start a run that learns.

No threshold is built in: it is a judgment from these columns.


## Sweep: reading a table (invented)

Simulated: two invented checkpoints, 64 prompts.

| checkpoint | G | parse_rate | within_group_std_mean | frac_all_tied |
|:--|--:|--:|--:|--:|
| A | 4 | 0.89 | 0.29 | 0.44 |
| A | 8 | 0.90 | 0.36 | 0.17 |
| A | 16 | 0.91 | 0.38 | 0.09 |
| B | 4 | 0.53 | 0.04 | 0.92 |
| B | 8 | 0.52 | 0.04 | 0.92 |
| B | 16 | 0.54 | 0.04 | 0.89 |

> **Our reading: A can start a run at $G = 8$ or $16$; B cannot, and a larger $G$ does not repair it.**

- **A:** frac_all_tied falls 0.44 to 0.17 to 0.09 and the std rises 0.29 to 0.36 to 0.38
    - 8 to 16 doubles the cost for a smaller gain, so we would take 8
- **B:** parse_rate is about 0.53 and frac_all_tied stays near 0.89 even at $G = 16$
    - the fault is in the checkpoint: no-go, back to SFT


## Other ways: what else could decide it

> **Three other ways to decide, each with a cost. We did not test which is best (our reading).**

- **a short pilot run:** watch the first steps of the real run (decisions 1 and 3)
    - costs real GPU time
- **start with a small $G$ and raise it** if groups are mostly tied (decision 2)
    - each change complicates the run
- **an estimate from measured success rates** (decisions 1 and 2)
    - apply this session's formula to per-prompt rates from the exploration check
    - needs enough prompts
- any of these is acceptable if the decision is tied to the numbers


## Decision record: write what you decided and why

> **A decision record has four fields: Decision, Criterion, Evidence, Outcome. The Criterion says what would have made you decide otherwise.**

| field | what it holds | invented example (Decision 2) |
|:--|:--|:--|
| **Decision** | what you chose | use $G = 8$ |
| **Criterion** | what would have changed it | go to 16 if over 20% of groups were tied |
| **Evidence** | file and numbers | `readiness_sweep.jsonl`, row A, $G = 8$: frac_all_tied 0.17 |
| **Outcome** | what the run showed | `metrics.jsonl`: near 0.2 (invented); the criterion held |

- the report's heading is `## Pre-flight and decisions`, then `### Decision 1`, `### Decision 2`, `### Decision 3`
    - each has these fields and a `When:` line (before or after the run)
- the criterion is the example's own, not a recommended threshold
- records for Decisions 1 and 3, and one written after the run, are in the asides after the last slide


# What we established, and what it sets up

- **Established**
    - equal rewards in a group give no signal
    - a binary-reward group needs a hit and a miss: probability $1 - p^G - (1-p)^G$
    - we checked exploration skill by skill and chose the smallest $G$ with mostly informative groups
    - we read three sweep columns; other ways exist
    - a decision record makes each decision checkable, even when written afterwards
- **This session's limit**
    - tables and curves are invented or simulated
    - we did not test how well the pre-flight predicts a run
    - we claim no best recipe
- **Sets up**
    - Part 1, where you decide on your own checkpoint and $G$ with evidence tied to your numbers
    - the same signals appear in the run's log


## Aside: Decision records for Decisions 1 and 3 (invented)

The same four fields as Decision 2, with different evidence.

| field | Decision 1: is the checkpoint good enough? | Decision 3: go or no-go? |
|:--|:--|:--|
| **Decision** | good enough to start | go |
| **Criterion** | every skill the reward needs succeeds on at least 5% of samples; otherwise back to SFT | go if decisions 1 and 2 hold and parse_rate is at least 0.9 at the chosen $G$ |
| **Evidence** | exploration check (`base_rate.py`): extraction accuracy 0.93, chain-resolution rate 0.38 (invented) | sweep row A, $G = 8$: parse_rate 0.90 |
| **Outcome** | `grpo_signal.jsonl`: nonzero std from step 0 | the run finished; reward rose in `metrics.jsonl` |

- the thresholds are invented. Any rule is acceptable if tied to your numbers


## Aside: a record written after the run (invented)

> **A criterion written afterwards is labelled reconstructed, and must be a rule we could have applied to the evidence we had at the time.**

- **When:** after
- **Decision 2:** used $G = 8$
- **Criterion (reconstructed):** I wrote no rule before the run. Looking back, the rule is: take the smallest $G$ with frac_all_tied under 0.2
- **Evidence:** `grpo_signal.jsonl`, first 50 steps: frac_all_tied about 0.17 (invented)
- **Outcome:** `metrics.jsonl`: it stayed below 0.25 to the end, so the rule would have picked the same $G$
- acceptable because the record shows which numbers the decision rests on, and the label says the rule came later


## Aside: from a per-prompt $p$ to frac_all_tied

- for a binary reward, a prompt with success probability $p_j$ is all tied with probability $p_j^G + (1-p_j)^G$
- the expected frac_all_tied over prompts is the average of that quantity over the prompts
- a sweep and this session's table are two views of one formula. The sweep measures it; the table computes it
